In [ ]:
import pynq
import numpy as np
import csv

In [2]:
import csv
import numpy as np

with open('pt.csv') as f_pt, open('phi.csv') as f_phi:
    pt_reader, phi_reader = csv.reader(f_pt), csv.reader(f_phi)
    pt, phi, n = [], [], []

    for i, (row_pt, row_phi) in enumerate(zip(pt_reader, phi_reader)):
        if len(row_pt) != len(row_phi):
            raise ValueError(f"Different number of particles in event {i}: {len(row_pt)} vs {len(row_phi)}")
        pt.extend(map(float, row_pt))
        phi.extend(map(float, row_phi))
        n.append(len(row_pt))

pt = np.array(pt, dtype=np.float32)
phi = np.array(phi, dtype=np.float32)
n_particles = np.array(n, dtype=np.uint16)
n_events = len(n_particles)

In [3]:
overlay = pynq.Overlay('METAccelerator.bit')
METAccelerator = overlay.METAccelerator

ptBuffer = pynq.allocate(pt.shape, dtype=np.float32)
phiBuffer = pynq.allocate(phi.shape, dtype=np.float32)
nBuffer = pynq.allocate(n_particles.shape, dtype=np.uint16)
metBuffer = pynq.allocate(n_events, dtype=np.float32)

ptBuffer[:] = pt
phiBuffer[:] = phi
nBuffer[:] = n_particles

METAccelerator.write(METAccelerator.register_map.pt.address, ptBuffer.physical_address)
METAccelerator.write(METAccelerator.register_map.phi.address, phiBuffer.physical_address)
METAccelerator.write(METAccelerator.register_map.n_particles.address, nBuffer.physical_address)
METAccelerator.write(METAccelerator.register_map.n_events.address, n_events)

METAccelerator.write(METAccelerator.register_map.met.address, metBuffer.physical_address)

In [4]:
METAccelerator.write(METAccelerator.register_map.CTRL.AP_START, 1)
while not METAccelerator.register_map.CTRL.AP_IDLE:
    pass
met_pynq = np.asarray(metBuffer)

In [5]:
met_pynq

array([ 68.25,  73.75,  70.75,  62.  ,  39.5 ,  66.  ,  35.5 ,  47.25,
        62.25,   9.5 ,  56.75,  18.75,  71.5 ,  78.75,  83.75, 153.5 ,
        48.  , 124.5 ,  24.5 ,  60.5 ,  94.5 ,  32.25,  60.75,  44.5 ,
        63.5 , 108.25,  10.25,  23.5 ,  96.75,  54.25,  37.25,  19.25,
       120.25, 124.25,  15.75, 140.75,  35.75,  36.  , 165.75, 103.  ,
        15.  ,  60.5 ,  63.5 ,  84.25,  41.  ,  56.  , 121.75,  11.  ,
        45.75, 222.25,  27.75, 221.25, 102.25,  86.25,  43.75,  43.5 ,
        82.25,  68.  ,  33.5 ,  55.75,  94.75,  82.25,  77.5 , 119.  ,
        39.75,  76.25,  81.5 ,  49.25,  33.  ,  16.25,  55.75,  44.  ,
       119.25,  95.75,  52.5 ,  45.5 ,  62.  ,  57.5 ,  34.75, 114.25,
        39.5 ,  94.25,  47.  ,  75.  ,  51.5 ,  15.75,  46.5 ,  13.25,
        84.25,  64.75,  36.  ,  16.75,  25.  ,  17.75,  17.75,  16.75,
        43.75,  23.75, 102.  ,  19.  ,  12.75, 109.  ,  49.  ,  33.5 ,
        30.75,  23.25,  24.75,  25.25,  22.5 ,  92.25,  72.75, 131.5 ,
      

In [6]:
met_file = open('met_hls.csv')
met_reader = csv.reader(met_file)
met_csim = []
for row in met_reader:
    assert len(row) == 1, f"Only 1 MET expected per row, got {len(row)}"
    met_csim.append(float(row[0]))
met_csim = np.array(met_csim)

In [7]:
np.array_equal(met_csim, met_pynq)

True